### SImple GenAI app using langchain

In [1]:
import os
from dotenv import load_dotenv

## This function will load all the variable from .env file and will make them available
## os.environ directory (env_variablea)
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

## for langsmith tracking
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2'] = "true"
os.environ['LANGCHAIN_PROJECT'] = os.getenv("LANGCHAIN_PROJECT")

In [13]:
from langchain_openai import ChatOpenAI

openai_llm = ChatOpenAI(model='gpt-4o')  # use gpt-4o-mini for budget friendly approach
print(openai_llm)

metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17', 'langchain-openai': '1.6.0'}} profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x14532dfd0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x14532e270> root_client=<openai.OpenAI object at 0x1454f4410> root_async_client=<openai.AsyncOpenAI o

### Steps we will follow

1. load Data (Document)
2. Divide documents into chunks
3. text -> vectors (vector Embeddings)
4. Store in vectorestoreDB

#### Data Ingestion

In [2]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.langchain.com/langsmith/evaluation-types")
loader

/var/folders/0b/_fz3pln11xz8q6vfw486xw4r0000gn/T/ipykernel_25174/2366161791.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
/Users/spy/Desktop/agentic-AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/evaluation-types', 'title': 'Evaluation types - Docs by LangChain', 'language': 'en'}, page_content="Evaluation types - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageTestSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationEvaluation typesGet startedDatasets & ExperimentsEvaluatorsAnnotation QueuesTest from PlaygroundTest from StudioOverviewManage evaluatorsManage evaluators programmaticallySDKRun evaluators on experimentsEvaluator spendEvaluator typesUISDKFrameworks & integrationsRun evals with openevals packageRun evals with pytestRun evals with Vitest/JestHarbor integrationsImprove evaluatorsImprove LLM-as-j

### Chunking

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                               chunk_overlap=200)
documents = text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/evaluation-types', 'title': 'Evaluation types - Docs by LangChain', 'language': 'en'}, page_content="Evaluation types - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageTestSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationEvaluation typesGet startedDatasets & ExperimentsEvaluatorsAnnotation QueuesTest from PlaygroundTest from StudioOverviewManage evaluatorsManage evaluators programmaticallySDKRun evaluators on experimentsEvaluator spendEvaluator typesUISDKFrameworks & integrationsRun evals with openevals packageRun evals with pytestRun evals with Vitest/JestHarbor integrationsImprove evaluatorsImprove LLM-as-j

### Embeddings

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
db = FAISS.from_documents(documents, embeddings)

In [6]:
## querrying from vectorstoreDB
query = "Benchmarking"
result = db.similarity_search(query)
result[0].page_content

'\u200bBenchmarking\nBenchmarking compares multiple application versions on a curated dataset to identify the best performer. This process involves creating a dataset of representative inputs, defining performance metrics, and testing each version.\nBenchmarking requires dataset curation with gold-standard reference outputs and well-designed comparison metrics. Examples:\n\nRAG Q&A bot: Dataset of questions and reference answers, with an LLM-as-judge evaluator checking semantic equivalence between actual and reference answers.\nReAct agent: Dataset of user requests and reference tool calls, with a heuristic evaluator verifying all expected tool calls were made.'

In [14]:
## Retreival Chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the context provides:
    <context>
    {context}
    </context>
    """
)

document_chain = create_stuff_documents_chain(openai_llm, prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the context provides:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17', 'langchain-openai': '1.6.0'}}, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'au

In [18]:
from langchain_core.documents import Document
document_chain.invoke({
        'input':'Benchmarking',
        'context':[Document(page_content="""Offline evaluation types

Offline evaluation tests applications on curated datasets before deployment. By running evaluations on examples with reference outputs, teams can compare versions, validate functionality, and build confidence before exposing changes to users.
Run offline evaluations client-side using the LangSmith SDK (Python or TypeScript) or server-side via the Playground or by binding evaluators to a dataset.
Offline
​
Benchmarking

Benchmarking compares multiple application versions on a curated dataset to identify the best performer. This process involves creating a dataset of representative inputs, defining performance metrics, and testing each version.
Benchmarking requires dataset curation with gold-standard reference outputs and well-designed comparison metrics. Examples:
RAG Q&A bot: Dataset of questions and reference answers, with an LLM-as-judge evaluator checking semantic equivalence between actual and reference answers.
ReAct agent: Dataset of user requests and reference tool calls, with a heuristic evaluator verifying all expected tool calls were made.""")]
    })


'What are the types of offline evaluation mentioned in the provided context, and what do they involve?\n\nOffline evaluation in the provided context involves testing applications on curated datasets before they are deployed. This process allows teams to compare different versions of an application, validate their functionality, and build confidence in the changes before exposing them to users. Offline evaluations can be conducted client-side using the LangSmith SDK (in Python or TypeScript) or server-side through the Playground or by integrating evaluators with a dataset.\n\nBenchmarking, a type of offline evaluation, involves comparing multiple application versions using a curated dataset to identify which version performs best. This requires creating datasets with gold-standard reference outputs and establishing well-designed performance metrics for comparison. Benchmarking includes examples such as:\n\n- RAG Q&A bot: This involves a dataset with questions and reference answers, and 

**🔑 However, We want the documnets to first come from retrieve we just set up. That way, we can use the retriever to dynamically select the most relevant documnets and pass those in for a given query**

### Retriever

**A LangChain retriever is an interface that accepts a string query and returns relevant Document objects from a data source.** 

* **What is a Retriever?**
    - **Core Definition**: A component designed to find and return text chunks or documents based on user input.
    - **General Interface**: Unlike a vector store, a retriever does not strictly need to store documents; it only needs to return them.
    - **Runnable Integration**: Built on LangChain's standard Runnable interface, allowing smooth integration into chains and pipelines using .invoke() or .ainvoke()

In [19]:
from langchain_classic.chains import create_retrieval_chain

retriever = db.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)

retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x14178f230>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the context provides:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
     

In [22]:
## get the response from llm
response = retrieval_chain.invoke({'input':"benchmark"})
response

{'input': 'benchmark',
 'context': [Document(id='37cfac62-c80d-4a6d-b02f-3303315ce2ce', metadata={'source': 'https://docs.langchain.com/langsmith/evaluation-types', 'title': 'Evaluation types - Docs by LangChain', 'language': 'en'}, page_content='\u200bBenchmarking\nBenchmarking compares multiple application versions on a curated dataset to identify the best performer. This process involves creating a dataset of representative inputs, defining performance metrics, and testing each version.\nBenchmarking requires dataset curation with gold-standard reference outputs and well-designed comparison metrics. Examples:\n\nRAG Q&A bot: Dataset of questions and reference answers, with an LLM-as-judge evaluator checking semantic equivalence between actual and reference answers.\nReAct agent: Dataset of user requests and reference tool calls, with a heuristic evaluator verifying all expected tool calls were made.'),
  Document(id='b2f801f4-2251-491a-80d5-c636f2d4bb17', metadata={'source': 'https:

In [21]:
response['answer']

'What is benchmarking in the context provided?\n\nBenchmarking in the provided context refers to the process of comparing multiple application versions using a curated dataset to identify the best performing version. This involves creating a dataset with representative inputs, defining performance metrics, and testing each version against these metrics. The process requires careful curation of the dataset to include gold-standard reference outputs and well-designed comparison metrics. Examples include using a RAG Q&A bot or a ReAct agent with specific datasets and evaluators to assess performance. Common use cases involve evaluating factual accuracy offline and checking for toxicity in production responses online.'

In [23]:
response['context']

[Document(id='37cfac62-c80d-4a6d-b02f-3303315ce2ce', metadata={'source': 'https://docs.langchain.com/langsmith/evaluation-types', 'title': 'Evaluation types - Docs by LangChain', 'language': 'en'}, page_content='\u200bBenchmarking\nBenchmarking compares multiple application versions on a curated dataset to identify the best performer. This process involves creating a dataset of representative inputs, defining performance metrics, and testing each version.\nBenchmarking requires dataset curation with gold-standard reference outputs and well-designed comparison metrics. Examples:\n\nRAG Q&A bot: Dataset of questions and reference answers, with an LLM-as-judge evaluator checking semantic equivalence between actual and reference answers.\nReAct agent: Dataset of user requests and reference tool calls, with a heuristic evaluator verifying all expected tool calls were made.'),
 Document(id='b2f801f4-2251-491a-80d5-c636f2d4bb17', metadata={'source': 'https://docs.langchain.com/langsmith/evalu